[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/mlops-internship-in-a-book/blob/main/notebooks/week8_deployment_patterns_STARTER.ipynb)


[![Get the Book on Selar](https://img.shields.io/badge/Get%20the%20full%20book-Selar-1E7A34?style=for-the-badge)](https://selar.com/7m27877571)

# Week 8 -- Deployment Patterns (STARTER)
### The MLOps Internship | DataFlow Infrastructure

**Dataset:** CreditDefault-NG (UCI Credit Card Default)

**This week:** FastAPI model serving, Docker containerisation, blue-green deployment, canary routing

STARTER -- complete each exercise before moving on.


In [ ]:
# -- Colab Setup (skip if running locally) -------------------------
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/mlops-internship-in-a-book.git mlops-book
    %cd mlops-book
    !pip install -r requirements.txt -q
os.makedirs('outputs', exist_ok=True)
os.makedirs('models',  exist_ok=True)
os.makedirs('datasets', exist_ok=True)
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds ------------------------------------------
# SEED=42 ensures every random operation gives the same result on every run.
# Set seeds BEFORE any data loading, model creation, or training.
import random, numpy as np
SEED = 42
random.seed(SEED)       # Python random
np.random.seed(SEED)    # NumPy random (used by sklearn)
print(f'Seeds fixed: {SEED}')


## Learning Objectives

- Serve the credit model via FastAPI with /health, /predict, and /metrics endpoints
- Containerise the model server with Docker and verify identical predictions inside and outside the container
- Implement blue-green deployment with nginx traffic switching under 60 seconds
- Implement canary routing: 5% to new model, 95% to current model
- Verify rollback completes within the 60-second SLA using a timed script



---

## MONDAY -- "FastAPI Model Server"


FastAPI validates request schemas with Pydantic, generates automatic API documentation, and handles concurrent requests. Three endpoints required: /health (is the model loaded and responding?), /predict (return a prediction with a decision), and /metrics (Prometheus-format metrics for Week 9).

Pause and Predict -- what should the /health endpoint verify? Just 'is the server running', or should it also confirm the model can process a test prediction? What is the difference between those two health checks in terms of false positives (server up but model broken)?


### Exercise 8.1 -- FastAPI endpoint

Implement /predict with Pydantic validation. Test: 10 valid requests return status 200 and decision in APPROVE/REVIEW/DECLINE. One request with PAY_0=99 returns status 422.


In [ ]:
# Exercise 8.1: FastAPI endpoint
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Monday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Monday: FastAPI Model Server --
from fastapi import FastAPI
from pydantic import BaseModel, validator
import mlflow.sklearn, time

app   = FastAPI(title='CreditDefault-NG API', version='3.0.0')
model = mlflow.sklearn.load_model('models:/credit-default-v2/Production')

class Application(BaseModel):
    customer_id: int
    LIMIT_BAL:   float
    PAY_0:       int
    PAY_2:       int
    BILL_AMT1:   float
    PAY_AMT1:    float

    @validator('LIMIT_BAL')
    def limit_positive(cls, v):
        if v <= 0: raise ValueError('LIMIT_BAL must be positive')
        return v

    @validator('PAY_0')
    def pay0_valid_code(cls, v):
        if not -2 <= v <= 8: raise ValueError('PAY_0 must be a valid bureau code')
        return v

@app.get('/health')
def health():
    # Verify the model can process a test prediction, not just that the server is up
    try:
        model.predict_proba([[100000, 0, 0, 5000, 2000]])
        return {'status': 'healthy', 'model_version': '3.0.0'}
    except Exception as e:
        from fastapi import HTTPException
        raise HTTPException(status_code=503, detail=f'Model not responding: {e}')

@app.post('/predict')
def predict(app: Application):
    X    = [[app.LIMIT_BAL, app.PAY_0, app.PAY_2, app.BILL_AMT1, app.PAY_AMT1]]
    prob = model.predict_proba(X)[0][1]
    decision = 'APPROVE' if prob < 0.35 else 'REVIEW' if prob < 0.60 else 'DECLINE'
    return {'customer_id': app.customer_id, 'default_probability': round(float(prob), 4), 'decision': decision}


### Expected Output (illustrative — uvicorn + curl)

```
$ curl http://localhost:8000/health
{"status":"healthy","model_version":"3.0.0"}

$ curl -X POST http://localhost:8000/predict -d '{"customer_id":18442,"LIMIT_BAL":150000,
  "PAY_0":0,"PAY_2":0,"BILL_AMT1":45000,"PAY_AMT1":12000}'
{"customer_id":18442,"default_probability":0.1823,"decision":"APPROVE"}
```
Note the `/health` endpoint runs an actual prediction, not just a liveness check — a model
that fails to predict but still returns 200 on a naive health check is exactly the Week 0
failure mode this book opens with.


### Monday Morning Moment

*Slack -- Monday, 9:15am.*

**Sola Fashola:** Blue-green or canary first?

**Temi Adeyemi:** Blue-green first to confirm the container is healthy before any real traffic touches it. Canary after.

**Sola Fashola:** Canary failure criterion?

**Temi Adeyemi:** Approval rate deviation greater than 3pp from blue, or any model test failure in the CI/CD suite.

**Dr. Emeka Obi:** Write that into the runbook. It should not live only in your head.




---

## TUESDAY -- "Docker Containerisation"


Docker packages the model server and all dependencies into a container that runs identically on any machine -- your laptop, staging, and production. The Dockerfile defines: base image, dependency installation, code copy, health check, and startup command.


### Exercise 8.2 -- Docker build and verify

Build the image. Run it. Send a /predict request. Compare response to the non-containerised server for the same input. Debug until they match exactly.


In [ ]:
# Exercise 8.2: Docker build and verify
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Tuesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Tuesday: Docker Containerisation --
# Dockerfile
FROM python:3.10-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY serve/ ./serve/
COPY models/ ./models/
# HEALTHCHECK: Docker restarts the container if this fails
HEALTHCHECK --interval=30s --timeout=10s \
    CMD curl -f http://localhost:8000/health || exit 1
CMD ["uvicorn", "serve.app:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]

# Build: docker build -t creditdefault:v3 .
# Run:   docker run -p 8000:8000 creditdefault:v3
# Test:  curl http://localhost:8000/health


### Expected Output (illustrative — docker build/run)

```
$ docker build -t creditdefault:v3 .
[+] Building 24.6s
 => [internal] load build definition from Dockerfile
 => => naming to docker.io/library/creditdefault:v3

$ docker run -p 8000:8000 creditdefault:v3
INFO:     Uvicorn running on http://0.0.0.0:8000
INFO:     Application startup complete.
```
The `HEALTHCHECK` directive means `docker ps` will show `(healthy)` once `/health` starts
returning 200 — and Docker will automatically restart the container if it later starts
failing, without anyone needing to page on-call for that specific failure mode.



---

## WEDNESDAY -- "Blue-Green Deployment"


Blue-green runs two containers simultaneously: blue (current production, v2) and green (new model, v3). When green is ready and tested, nginx switches all traffic from blue to green. Rollback is the reverse switch. Both containers must be running before any switch.


### Exercise 8.3 -- Blue-green switch

Run both containers. Send 100 requests to blue. Switch to green. Verify all 100 subsequent requests go to green. Measure switch latency.


In [ ]:
# Exercise 8.3: Blue-green switch
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Wednesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Wednesday: Blue-Green Deployment --
# nginx blue-green config
upstream credit_blue  { server localhost:8000; }  # v2 current
upstream credit_green { server localhost:8001; }  # v3 new

server {
    listen 80;
    location /predict {
        proxy_pass http://credit_blue;  # change to credit_green to deploy
        add_header X-Model-Version 'blue';
    }
}
# Deploy: sed -i 's/credit_blue/credit_green/' nginx.conf && nginx -s reload
# Rollback: reverse, reload


### Expected Output (illustrative — nginx reload)

```
$ sed -i 's/credit_blue/credit_green/' nginx.conf && nginx -s reload
$ curl -sI http://localhost/predict | grep X-Model-Version
X-Model-Version: green
```
Blue-green is all-or-nothing: 100% of traffic moves in one atomic switch. Compare this to
Thursday's canary pattern, which moves traffic gradually — the trade-off is blast radius
(canary limits it; blue-green doesn't) versus complexity (canary needs weighted routing;
blue-green doesn't).



---

## THURSDAY -- "Canary Routing"


Canary routing sends 5% of traffic to the new model. Nginx weighted upstream implements this: weight=95 for blue, weight=5 for green. If the new model behaves correctly for 24 hours, increase to 20%, then 50%, then 100%.

Pause and Predict -- what is your failure criterion for aborting the canary? State a specific numeric threshold for approval rate deviation before reading the code.


### Exercise 8.4 -- Canary validation script

Send 200 requests with canary active. Parse X-Served-By headers to separate blue vs green responses. Compute approval rate per container. Flag if green deviates from blue by more than 3pp.


In [ ]:
# Exercise 8.4: Canary validation script
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Thursday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Thursday: Canary Routing --
# Nginx canary config
upstream credit_canary {
    server localhost:8000 weight=95;  # blue: 95%
    server localhost:8001 weight=5;   # green: 5%
}
server {
    listen 80;
    location /predict {
        proxy_pass http://credit_canary;
        add_header X-Served-By $upstream_addr;  # for monitoring
    }
}
# Promotion: Day 1: 5% green. Day 2: 20%. Day 3: 50%. Day 4: 100%.
# Abort: if green approval rate differs from blue by > 3pp, set weight=0 and reload


### Expected Output (illustrative — nginx canary weights)

```
$ curl -sI http://localhost/predict | grep X-Served-By
X-Served-By: 127.0.0.1:8001   # (~5% of requests -- green/canary)
X-Served-By: 127.0.0.1:8000   # (~95% of requests -- blue/stable)
```
Over ~20 requests you should see roughly 1 hit `8001` and 19 hit `8000` — if the ratio looks
very different, check the `weight=` values before assuming something else is wrong.



---

## FRIDAY -- "The Friday Build: Rollback Timer"


Deploy v3 as a canary (5%). Send 50 test requests. Execute the rollback script. Measure time from command to 100% traffic on blue. SLA: 60 seconds. Record the result.


**Friday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Friday: The Friday Build: Rollback Timer --
import subprocess, time, requests

start = time.time()

# Switch green weight to 0
subprocess.run(['sed', '-i',
    's/server localhost:8001 weight=5/server localhost:8001 weight=0/',
    '/etc/nginx/sites-available/credit-model'])
subprocess.run(['nginx', '-s', 'reload'])

# Verify all traffic on blue
resp = requests.get('http://localhost/health')
elapsed = time.time() - start
print(f'Rollback: {elapsed:.1f}s')
assert elapsed < 60, f'Rollback SLA breach: {elapsed:.1f}s'    # this IS the SLA test, not a formality


### Expected Output (illustrative)

```
Rollback: 2.3s
```
The `assert` on the last line is the actual pass/fail gate for this week's core deliverable —
if your rollback script takes 61 seconds instead of 60, this line is what should fail loudly,
not a human noticing later that something felt slow.


### Friday Workplace Moment

*DataFlow -- Friday standup.*

**Dr. Emeka Obi:** Rollback time.

**Temi Adeyemi:** 13 seconds. nginx reload is the bottleneck at 9 seconds.

**Sola Fashola:** What if nginx fails?

**Temi Adeyemi:** Script checks the exit code. Non-zero triggers PagerDuty. On-call takes manual control. Blue stays live until they do.

**Dr. Emeka Obi:** Canary metrics?

**Temi Adeyemi:** Approval rate: 34.3% green vs 34.0% blue -- 0.3pp difference, within the 3pp threshold.

**Dr. Emeka Obi:** Promotion review Monday. Sola signs off.



## YOUR WEEK 8 CHECKLIST

Check each item before moving on.

- [ ] FastAPI validates all inputs and returns 422 for invalid bureau codes.
- [ ] Docker container produces identical predictions to the non-containerised server.
- [ ] Blue-green switch moves all traffic in under 2 seconds and is fully scripted.
- [ ] Canary routes approximately 5% to the new model, verified by header analysis.
- [ ] Rollback completes within 60 seconds and is measured by a timer test.


## Challenge Task

> **Core path:** Implement request logging middleware: log every prediction to PostgreSQL with customer_id, timestamp, LIMIT_BAL, default_probability, decision, model_version, and latency_ms.

> **Advanced path:** Build shadow mode: route 100% to blue, mirror requests to green, log green's predictions but do not serve them. Compare approval rates after 24 hours.


## Common Mistakes

**Deploying without canary:** Even a model passing all staging tests can fail on production traffic. Canary limits blast radius to 5%.

**Rollback requiring manual SSH:** Rollback requiring 10 minutes of manual config editing is a recovery procedure, not a rollback. Script it. Time it.

**Health check that only confirms the server is running:** A health check returning 200 when the model failed to load is worse than no health check.



---

**The MLOps Internship** -- Book 4 of 9, InternshipInABook™

Dataset: CreditDefault-NG | Company: DataFlow Infrastructure, Lagos/Abuja

[internshipinabook.com](https://internshipinabook.com)


📘 **Get the complete illustrated book (all 13 weeks, DOCX + EPUB):** [https://selar.com/7m27877571](https://selar.com/7m27877571)